# Final ResNet-18 training reproduction

This readable runner reproduces the final 3+7 epoch training procedure. Cells are deliberately unexecuted: publishing did not rerun training. Executed final evidence is in emotion_final_results.ipynb and reports/emotion/final. It uses the original sample manifest from Drive to preserve exact training record order; no full manifest or raw image is committed. This creates a NEW run and does not overwrite the tested epoch-10 checkpoint. It never evaluates the test split.

## Prerequisites
Use the agreed project environment and extract the submission source into /content/facial-attendance. Mount the Google account holding the approved dataset archive and original run folder. No Kaggle API, dataset reconstruction, or embedded credentials are used.

In [ ]:
import os
import shutil
import sys
from datetime import UTC, datetime
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Connect a GPU runtime."
from google.colab import drive  # noqa: E402 — Colab import follows explicit GPU preflight.

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
repo = Path("/content/facial-attendance")
assert (repo / "scripts/emotion_final/train.py").is_file(), (
    "Extract the submission source under /content/facial-attendance first."
)
sys.path.insert(0, str(repo / "scripts/emotion_final"))
from load_cleaned_zip import prepare_dataset  # noqa: E402 — import from the selected source checkout.

drive_root = Path("/content/drive/MyDrive/facial-attendance/emotion")
data = prepare_dataset(
    drive_root / "sad-happy-neutral-geometry-v2-20261121.zip",
    Path("/content/emotion-datasets"),
)
original = drive_root / "resnet-sad-happy-neutral-geometry-v2-20261121T073048197978Z"
assert original.is_dir(), "The original run records must be available in mounted Drive."
RUN = drive_root / (
    "emotion-final-reproduction-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
)
RUN.mkdir(exist_ok=False)
for name in ["sample-manifest.csv", "planned-configuration.json"]:
    shutil.copyfile(original / name, RUN / name)
for source, target in [
    ("train.py", "train_full_gpu.py"),
    ("model_code.py", "model_code.py"),
    ("plot.py", "plot.py"),
]:
    shutil.copyfile(repo / "scripts/emotion_final" / source, RUN / target)
os.environ["EXPRESSION_RAW_ROOT"] = str(data)
print("NEW training run:", RUN)

In [ ]:
import subprocess

subprocess.run([sys.executable, str(RUN / "train_full_gpu.py")], cwd=RUN, check=True)

In [ ]:
import subprocess

from IPython.display import Image, display

subprocess.run([sys.executable, str(RUN / "plot.py")], cwd=RUN, check=True)
display(Image(filename=str(RUN / "validation-results.png")))